In [ ]:
# Install dependencies
!pip install sdv openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality

# =============================================================================
# CONFIGURATION
# =============================================================================
# Try to find any Excel file if the default path fails
REAL_DATA_PATH = "REAL DATA.xlsx"
N_SYNTHETIC = 274

if not os.path.exists(REAL_DATA_PATH):
    excel_files = glob.glob("*.xlsx")
    if excel_files:
        # Skip the output file if it already exists
        valid_files = [f for f in excel_files if f != 'synthetic_survey_data.xlsx']
        if valid_files:
            REAL_DATA_PATH = valid_files[0]
            print(f"Target file not found. Auto-detected data source: {REAL_DATA_PATH}")

COLUMN_NAMES = {
    "age":         "Q1. Age Group",
    "gender":      "Q2. Gender",
    "frequency":   "Q3. Shopping Frequency",
    "quality":     "Q4. Product Quality Satisfaction",
    "experience":  "Q5. Shopping Experience Rating",
    "delivery":    "Q6. Delivery Service Speed",
    "service":     "Q7. Customer Service Satisfaction",
    "issues":      "Q8. Issues with Luxury Shop Services",
    "recommend":   "Q9. Would Recommend Luxury Products",
    "reason":      "Q10. Main Reason for Choosing Luxury",
    "shop_again":  "Q11. Plan to Shop Again in Future",
}

ORDINAL_ORDERS = {
    COLUMN_NAMES["age"]:        ["Under 18", "18–25", "26–35", "36–45"],
    COLUMN_NAMES["frequency"]:  ["This is my first time", "Occasionally", "Regularly", "Very frequently"],
    COLUMN_NAMES["quality"]:    ["Very dissatisfied", "Dissatisfied", "Neutral", "Satisfied", "Very satisfied"],
    COLUMN_NAMES["experience"]: ["Strongly Disagree", "Disagree", "Neutral", "Agree", "Strongly Agree"],
    COLUMN_NAMES["delivery"]:   ["Slow", "Average", "Fast", "Very Fast"],
    COLUMN_NAMES["service"]:    ["Very dissatisfied", "Dissatisfied", "Neutral", "Satisfied", "Very satisfied"],
    COLUMN_NAMES["issues"]:     ["Never", "Rarely", "Yes, sometimes", "Yes, frequently"],
    COLUMN_NAMES["recommend"]:  ["Definitely not", "Probably not", "Probably", "Definitely"],
    COLUMN_NAMES["shop_again"]: ["No", "Probably not", "Maybe", "Yes, definitely"],
}
NOMINAL_COLS = [COLUMN_NAMES["gender"], COLUMN_NAMES["reason"]]

# STEP 2 — Load your uploaded data
if os.path.exists(REAL_DATA_PATH):
    print(f"Loading your real data from {REAL_DATA_PATH}...")
    real_data = pd.read_excel(REAL_DATA_PATH)
else:
    raise FileNotFoundError(f"Could not find {REAL_DATA_PATH}. Please ensure it is uploaded to the sidebar.")

# STEP 3 — Metadata Detection
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(real_data)
for col in list(ORDINAL_ORDERS.keys()) + NOMINAL_COLS:
    if col in real_data.columns:
        metadata.update_column(column_name=col, sdtype="categorical")

# STEP 4 — Fit Synthesizer
print("Fitting GaussianCopula to your real data distributions...")
synthesizer = GaussianCopulaSynthesizer(metadata, enforce_min_max_values=True, enforce_rounding=True)
synthesizer.fit(real_data)

# STEP 5 — Sample
synthetic_data = synthesizer.sample(num_rows=N_SYNTHETIC)
synthetic_data.to_excel("synthetic_survey_data.xlsx", index=False)

# STEP 6 — Quality Report
quality_report = evaluate_quality(real_data, synthetic_data, metadata)
print(f"Overall quality score: {quality_report.get_score():.3f}")

Loading your real data from REAL DATA.xlsx...
Fitting GaussianCopula to your real data distributions...


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1674.49it/s]|
Column Shapes Score: 96.57%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 229.57it/s]|
Column Pair Trends Score: 73.48%

Overall Score (Average): 85.02%

Overall quality score: 0.850


### Preview of Synthetic Data
Below are the first 5 rows of the generated synthetic survey responses.

In [ ]:
# Display the first few rows
display(synthetic_data.head())

# Provide a download link for the Excel file
from google.colab import files
print("\nClick the link below to download the full dataset:")
files.download('synthetic_survey_data.xlsx')

### Preview of Synthetic Data
Below are the first 5 rows of the generated synthetic survey responses.

In [ ]:
# Display the first few rows
display(synthetic_data.head())

# Provide a download link for the Excel file
from google.colab import files
print("\nClick the link below to download the full dataset:")
files.download('synthetic_survey_data.xlsx')

,Q1. Age Group,Q2. Gender,Q3. Shopping Frequency,Q4. Product Quality Satisfaction,Q5. Shopping Experience Rating,Q6. Delivery Service Speed,Q7. Customer Service Satisfaction,Q8. Issues with Luxury Shop Services,Q9. Would Recommend Luxury Products,Q10. Main Reason for Choosing Luxury,Q11. Plan to Shop Again in Future
0,36–45,Female,Very frequently,Dissatisfied,1,Slow,Dissatisfied,Never,Probably not,Stylish design,Maybe
1,18–25,Male,Occasionally,Very satisfied,3,Very fast,Very satisfied,"Yes, frequently",Definitely,Product quality,"Yes, definitely"
2,18–25,Male,Occasionally,Very satisfied,1,Very fast,Very satisfied,"Yes, sometimes",Definitely,Customer service,"Yes, definitely"
3,36–45,Female,Regularly,Dissatisfied,3,Average,Dissatisfied,Rarely,Definitely not,Stylish design,No
4,18–25,Male,Regularly,Satisfied,1,Fast,Very satisfied,Never,Probably not,Customer service,Maybe



Click the link below to download the full dataset:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>